# Verify published FACED DE features
Recompute all 123 subjects from the published preprocessed EEG archive.


In [1]:
import hashlib
import json
import pickle
import time
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import mne
from threadpoolctl import threadpool_limits

root = Path('C:/FACED dataset')
work = Path('D:/EEG Experiment Artifacts/2026-11-02/reproduction')
archive = root/'Other Files/Processed_data.zip'
output = work/'full_de_verification'
output.mkdir(exist_ok=True)
bands = [(1,4),(4,8),(8,14),(14,30),(30,47)]
assert mne.__version__ == '1.2.1'
rows = []
started = time.perf_counter()
with zipfile.ZipFile(archive) as z:
    for sub in range(123):
        saved = output/f'sub{sub:03d}.json'
        if saved.exists():
            rows.extend(json.loads(saved.read_text()))
            continue
        subject_start = time.perf_counter()
        member = z.getinfo(f'Processed_data/sub{sub:03d}.pkl')
        with z.open(member) as f:
            data = pickle.load(f)
        assert data.shape == (28,32,7500)
        path = root/'EEG_Features/DE'/f'sub{sub:03d}.pkl.pkl'
        released = pickle.load(open(path,'rb'))
        subject_rows = []
        with threadpool_limits(limits=1):
            for band,(low,high) in enumerate(bands):
                # Batch independent channels/trials; temporal filtering is unchanged.
                filtered = mne.filter.filter_data(data[:,:30].reshape(840,7500),250,
                                                  l_freq=low,h_freq=high,n_jobs=1,verbose=False)
                de = .5*np.log(2*np.pi*np.e*filtered.reshape(28,30,30,250).var(axis=-1))
                delta = de-released[:,:30,:,band]
                subject_rows.append({'subject':sub,'band':band,'max_absolute_error':float(np.max(np.abs(delta))),
                                     'mean_absolute_error':float(np.mean(np.abs(delta))),
                                     'archive_member_crc':member.CRC,'released_de_sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
                assert np.isfinite(de).all()
        saved.write_text(json.dumps(subject_rows,indent=2))
        rows.extend(subject_rows)
        print('Verified subject',sub,'max error',max(x['max_absolute_error'] for x in subject_rows),
              'seconds',round(time.perf_counter()-subject_start,2),flush=True)
        del data,released,filtered,de,delta
frame = pd.DataFrame(rows)
frame.to_csv(output/'all_subjects.csv',index=False)
result = {'subjects':123,'bands':5,'compared_features':123*28*30*30*5,
          'max_absolute_error':float(frame['max_absolute_error'].max()),'mne':mne.__version__,
          'numpy':np.__version__,'seconds':time.perf_counter()-started,
          'interpretation':'Recomputed from published preprocessed EEG with the released DE formula and MNE version; raw BDF artifact rejection was not rerun.'}
(output/'summary.json').write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))
assert result['max_absolute_error']<1e-10


Verified subject 0 max error 0.0 seconds 1.91
Verified subject 1 max error 0.0 seconds 1.55
Verified subject 2 max error 0.0 seconds 1.51
Verified subject 3 max error 0.0 seconds 1.5
Verified subject 4 max error 0.0 seconds 1.47
Verified subject 5 max error 0.0 seconds 1.47
Verified subject 6 max error 0.0 seconds 1.47
Verified subject 7 max error 0.0 seconds 1.47
Verified subject 8 max error 0.0 seconds 1.43
Verified subject 9 max error 0.0 seconds 1.42
Verified subject 10 max error 0.0 seconds 1.43
Verified subject 11 max error 0.0 seconds 1.43
Verified subject 12 max error 0.0 seconds 1.42
Verified subject 13 max error 0.0 seconds 1.43
Verified subject 14 max error 0.0 seconds 1.45
Verified subject 15 max error 0.0 seconds 1.44
Verified subject 16 max error 0.0 seconds 1.42
Verified subject 17 max error 0.0 seconds 1.39
Verified subject 18 max error 0.0 seconds 1.4
Verified subject 19 max error 0.0 seconds 1.4
Verified subject 20 max error 0.0 seconds 1.4
Verified subject 21 max err